# Role Adherence Metric Example

This notebook demonstrates how to use the **RoleAdherence** metric from Gaussia to evaluate whether an AI assistant consistently adheres to its defined role across a multi-turn conversation.

The metric implements the formula:

$$\text{RoleAdherence}(R, T) = \frac{1}{n} \sum_{i=1}^{n} \text{adhere}(t_i, T_{<i}, R)$$

Where each turn $t_i$ is evaluated using the full conversation history $T_{<i}$ and the role definition $R$ — without requiring ground-truth reference responses.

### Dataset

The example uses a **FinTrack** digital banking support scenario with two sessions:

| Session | Description |
|---|---|
| `session_adherent_001` | All 4 turns fully adhere to the role (scope, tone, constructive behavior) |
| `session_violations_002` | 4 turns mixing adherent responses with a scope violation (investment advice), a tone violation (dismissive language), and a constructive failure (no next steps offered) |

## Installation

Este proyecto usa `uv`. La celda siguiente instala `langchain-groq` en el venv del repo.

Si preferís instalarlo desde la terminal antes de abrir Jupyter:

```bash
uv add langchain-groq
```

In [1]:
!uv add langchain-groq -q

## Setup

Import the required modules and configure your API key.

In [ ]:
import sys
from pathlib import Path

# Add examples directory to path for helpers import
sys.path.insert(0, str(Path("../..").resolve()))

from helpers.retriever import LocalRetriever
from langchain_openai import ChatOpenAI

from gaussia.metrics.role_adherence import LLMJudgeStrategy, RoleAdherence

In [ ]:
import getpass

OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")

## Initialize the Judge Model

The `LLMJudgeStrategy` uses any LangChain-compatible chat model that **exposes first-token logprobs** (OpenAI, Azure OpenAI, Ollama, LiteLLM, or HuggingFace TGI via the OpenAI-compatible class). The judge asks a binary YES/NO question and converts the first-token probabilities into a calibrated `[0, 1]` adherence score using `P(YES) / (P(YES) + P(NO))`.

Anthropic, Gemini and Bedrock are **not** compatible — they do not return logprobs.

In [ ]:
judge_model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
)

scoring_strategy = LLMJudgeStrategy(
    model=judge_model,
    verbose=True,
)

## Run the Role Adherence Metric

The metric reads `chatbot_role` from each session in the dataset and evaluates every turn in context of the full prior conversation history.

Key parameters:
- `binary=True` — per-turn score is binarized at `threshold`; session score = proportion of adherent turns. Set `False` to keep the raw continuous logprob score and average those.
- `strict_mode=False` — session is adherent if `mean >= threshold` (not all-or-nothing)
- `threshold=0.5` — minimum score to classify a turn (or the session) as adherent

In [ ]:
metrics = RoleAdherence.run(
    LocalRetriever,
    dataset_path="dataset_role_adherence.json",
    scoring_strategy=scoring_strategy,
    binary=True,
    strict_mode=False,
    threshold=0.5,
    verbose=True,
)

## Session-Level Results

Each `RoleAdherenceMetric` contains:
- `role_adherence`: proportion of turns the assistant adhered to the role (in binary mode)
- `adherent`: whether the session passes the threshold overall
- `turns`: per-turn breakdown with individual scores

> **Note:** the judge does not return a textual reason — the score is derived directly from the first-token probabilities. Inspecting the underlying logprobs requires calling `Judge.check_logprob_binary()` directly.

In [6]:
for metric in metrics:
    status = "✓ ADHERENT" if metric.adherent else "✗ NON-ADHERENT"
    print(f"Session: {metric.session_id}")
    print(f"  Role Adherence Score: {metric.role_adherence:.2f}  [{status}]")
    print(f"  Turns evaluated: {metric.n_turns}")
    print()

Session: session_adherent_001
  Role Adherence Score: 1.00  [✓ ADHERENT]
  Turns evaluated: 4

Session: session_violations_002
  Role Adherence Score: 0.50  [✓ ADHERENT]
  Turns evaluated: 4



## Per-Turn Breakdown

Drill into each turn to see where violations occurred and why the judge flagged them.

In [ ]:
for metric in metrics:
    print(f"=== {metric.session_id} ===")
    for turn in metric.turns:
        adherent_label = "✓" if turn.adherent else "✗"
        print(f"  [{adherent_label}] Turn {turn.qa_id}  score={turn.adherence_score:.2f}")
    print()

## Summary Statistics

In [8]:
total_turns = sum(m.n_turns for m in metrics)
adherent_turns = sum(t.adherent for m in metrics for t in m.turns)
adherent_sessions = sum(1 for m in metrics if m.adherent)
avg_score = sum(m.role_adherence for m in metrics) / len(metrics)

print(f"Sessions evaluated : {len(metrics)}")
print(f"Adherent sessions  : {adherent_sessions}/{len(metrics)}")
print(f"Turns evaluated    : {total_turns}")
print(f"Adherent turns     : {adherent_turns}/{total_turns}")
print(f"Average score      : {avg_score:.2f}")

Sessions evaluated : 2
Adherent sessions  : 2/2
Turns evaluated    : 8
Adherent turns     : 6/8
Average score      : 0.75


## Continuous Mode

Pass `binary=False` to `RoleAdherence` to skip the threshold step and average the raw continuous logprob scores per turn. The judge always asks a YES/NO question; `binary` only controls whether the per-turn score gets rounded against `threshold`.

In [ ]:
metrics_continuous = RoleAdherence.run(
    LocalRetriever,
    dataset_path="dataset_role_adherence.json",
    scoring_strategy=scoring_strategy,
    binary=False,
    threshold=0.5,
)

for metric in metrics_continuous:
    print(f"{metric.session_id}: role_adherence={metric.role_adherence:.3f}")
    for turn in metric.turns:
        print(f"  {turn.qa_id}: {turn.adherence_score:.3f}")

## Strict Mode

With `strict_mode=True`, a session is only marked `adherent=True` if **every single turn** passes. Useful for high-compliance scenarios where a single violation is unacceptable.

In [ ]:
metrics_strict = RoleAdherence.run(
    LocalRetriever,
    dataset_path="dataset_role_adherence.json",
    scoring_strategy=scoring_strategy,
    binary=True,
    strict_mode=True,
    threshold=0.5,
)

for metric in metrics_strict:
    status = "PASS" if metric.adherent else "FAIL"
    print(f"{metric.session_id}: {status}  (score={metric.role_adherence:.2f})")